In [1]:
# load mortality 

import pandas as pd

# Read the death parameters CSV file into a pandas DataFrame
death_params_df = pd.read_csv("../data/death_params.csv")



Chinese: 84.3 years
Indian: 81.3 years
Malay: 79.4 years 

https://www.moh.gov.sg/newsroom/current-life-expectancy-by-racial-groups/

this is the target 

In [2]:
death_params_df

,sim_year,agent_gender,agent_age,agent_death_prob
0,1990,female,0,0.0062
1,1990,female,1,0.0002
2,1990,female,2,0.0002
3,1990,female,3,0.0002
4,1990,female,4,0.0002
...,...,...,...,...
10487,2050,male,81,0.0595
10488,2050,male,82,0.0595
10489,2050,male,83,0.0595
10490,2050,male,84,0.0595


In [3]:
def get_agent_death_prob(sim_year, agent_gender, agent_age, df=death_params_df):
    """
    Given sim_year, agent_gender, agent_age, return agent_death_prob.
    Args:
        sim_year (int): Simulation year
        agent_gender (str): 'male' or 'female'
        agent_age (int): Age of the agent
        df (pd.DataFrame): DataFrame containing the mortality parameters. Default is death_params_df.
    Returns:
        float: agent_death_prob for the specified (year, gender, age)
    Raises:
        ValueError: If no matching entry found
    """
    row = df[
        (df['sim_year'] == sim_year) &
        (df['agent_gender'] == agent_gender) &
        (df['agent_age'] == agent_age)
    ]
    if not row.empty:
        return row.iloc[0]['agent_death_prob']
    


In [8]:
# this is for 1990 - 2024; 
import numpy as np
result_matrix = np.fromfile("../../data/bin/result_matrix_data.bin", dtype=np.float64)  # Specify data type if needed
result_matrix = result_matrix.reshape(8, 86, 35)  # Reshape to correct dimensions (8 cohorts, 86 ages, 35 years)


### focus on result_matrix(2:3,0:86,31)

they refer to male and female malay population of year 2021;
the last one stand for 85 and 85+


we have 
get_agent_death_prob(2021, agent_gender, agent_age, df=death_params_df)
to get mortality

please help to calculate life expectancy of malay over all ages and genders in 2021;




In [17]:
import numpy as np

def compute_life_expectancy(gender, death_params_df):
    """
    Compute life expectancy at birth for a given gender
    using death probabilities up to age 85, and use age 85 value for 85+.
    """

    # --------------------------
    # 1. Collect q_x for ages 0–85
    # --------------------------
    q = np.array([
        get_agent_death_prob(2021, gender, int(a), df=death_params_df)
        for a in range(0, 86)
    ])

    # --------------------------
    # 2. Extend mortality to age 120 using q[85]
    # --------------------------
    max_age = 120
    q_extended = np.zeros(max_age + 1)
    q_extended[:86] = q[:]            # 0..85
    q_extended[86:] = q[85]           # age 86..120 uses mortality at 85

    # --------------------------
    # 3. Build survival curve s[a]
    # --------------------------
    s = np.zeros(max_age + 1)
    s[0] = 1.0

    for a in range(1, max_age + 1):
        s[a] = s[a - 1] * (1 - q_extended[a])

    # --------------------------
    # 4. Life expectancy = sum of survival probabilities
    # --------------------------
    life_expectancy = np.sum(s)

    return life_expectancy


# --------------------------
# Compute for overall male & female
# --------------------------
LE_male   = compute_life_expectancy('male',   death_params_df)
LE_female = compute_life_expectancy('female', death_params_df)

print("Overall Male Life Expectancy (2021):   ", LE_male)
print("Overall Female Life Expectancy (2021): ", LE_female)

# --- Compute male & female life expectancy ---
LE_male   = compute_life_expectancy('male',   death_params_df)
LE_female = compute_life_expectancy('female', death_params_df)

# --- Get overall male/female population in 2021 ---
overall_male_pop   = result_matrix[2, :, 31].sum()
overall_female_pop = result_matrix[3, :, 31].sum()

N_tot = overall_male_pop + overall_female_pop

# --- Overall life expectancy ---
LE_overall = (overall_male_pop * LE_male + overall_female_pop * LE_female) / N_tot

print("Overall Male Life Expectancy (2021):      ", LE_male)
print("Overall Female Life Expectancy (2021):    ", LE_female)
print("Overall Life Expectancy (2021):           ", LE_overall)

Overall Male Life Expectancy (2021):    80.75540303117403
Overall Female Life Expectancy (2021):  85.77873382221327
Overall Male Life Expectancy (2021):       80.75540303117403
Overall Female Life Expectancy (2021):     85.77873382221327
Overall Life Expectancy (2021):            83.27152477688075



should use grid search here;

but for simplicity, we just try different beta "x/ 40.0"

In [18]:
import numpy as np


def compute_adjusted_life_expectancy(gender, death_params_df):
    """
    Life expectancy with adjusted mortality from age 40+:
    q*(age) = q(age) * age * beta
    with beta = 1.01 / 40
    """

    beta = 0.21 / 40.0
    max_age = 120

    # -- 1) get q for ages 0..85 --
    q = np.array([
        get_agent_death_prob(2021, gender, int(a), df=death_params_df)
        for a in range(0, 86)
    ])

    # -- 2) apply adjustment from age 40 onwards --
    q_adj = q.copy()

    for a in range(45, 86):             # 40..85
        q_adj[a] = q[a] * np.exp(a * beta)

    # -- 3) extend 85+ using q_adj[85] --
    q_extended = np.zeros(max_age + 1)
    q_extended[:86] = q_adj
    q_extended[86:] = q_adj[85]         # ages 86..120 use age=85 mortality

    # -- 4) build survival curve --
    s = np.zeros(max_age + 1)
    s[0] = 1.0

    for a in range(1, max_age + 1):
        s[a] = s[a - 1] * (1 - q_extended[a])

    # -- 5) life expectancy = sum of survival probabilities --
    LE = np.sum(s)
    return LE

LE_male_new   = compute_adjusted_life_expectancy('male',   death_params_df)
LE_female_new = compute_adjusted_life_expectancy('female', death_params_df)

print("Adjusted Malay Male LE:", LE_male_new)
print("Adjusted Malay Female LE:", LE_female_new)

mal_male_pop   = result_matrix[2, :, 31].sum()
mal_female_pop = result_matrix[3, :, 31].sum()
N_tot = mal_male_pop + mal_female_pop

LE_overall_new = (mal_male_pop * LE_male_new + mal_female_pop * LE_female_new) / N_tot

print("Adjusted Overall Malay Life Expectancy:", LE_overall_new)

Adjusted Malay Male LE: 77.13636855158391
Adjusted Malay Female LE: 81.819167400589
Adjusted Overall Malay Life Expectancy: 79.48192222998591


## indian

In [19]:
import numpy as np

def compute_adjusted_life_expectancy(gender, death_params_df):
    """
    Life expectancy with adjusted mortality from age 40+:
    q*(age) = q(age) * exp(a * beta)
    with beta = 0.21 / 40
    """
    beta = 0.1 / 40.0
    max_age = 120

    # -- 1) q(a) for 0..85 --
    q = np.array([
        get_agent_death_prob(2021, gender, int(a), df=death_params_df)
        for a in range(0, 86)
    ])

    # -- 2) adjust from age 40 onwards --
    q_adj = q.copy()
    for a in range(40, 86):  # 40..85
        q_adj[a] = q[a] * np.exp(a * beta)

    # -- 3) extend 85+ --
    q_extended = np.zeros(max_age + 1)
    q_extended[:86] = q_adj
    q_extended[86:] = q_adj[85]  # 86..120

    # -- 4) survival --
    s = np.zeros(max_age + 1)
    s[0] = 1.0
    for a in range(1, max_age + 1):
        s[a] = s[a - 1] * (1 - q_extended[a])

    # -- 5) life expectancy --
    LE = np.sum(s)
    return LE


# ----------------------------------------------------
# Compute **Indian** male and female adjusted life expectancy
# ----------------------------------------------------

LE_indian_male_new   = compute_adjusted_life_expectancy('male',   death_params_df)
LE_indian_female_new = compute_adjusted_life_expectancy('female', death_params_df)

print("Adjusted Indian Male LE:", LE_indian_male_new)
print("Adjusted Indian Female LE:", LE_indian_female_new)


# ----------------------------------------------------
# Population-weighted overall Indian life expectancy
# result_matrix indices:
# 4 = Indian male, 5 = Indian female
# ----------------------------------------------------

ind_male_pop   = result_matrix[4, :, 31].sum()
ind_female_pop = result_matrix[5, :, 31].sum()
N_tot_ind = ind_male_pop + ind_female_pop

LE_indian_overall_new = (
    ind_male_pop * LE_indian_male_new +
    ind_female_pop * LE_indian_female_new
) / N_tot_ind

print("Adjusted Overall Indian Life Expectancy:", LE_indian_overall_new)

Adjusted Indian Male LE: 78.91748585343822
Adjusted Indian Female LE: 83.75677408048108
Adjusted Overall Indian Life Expectancy: 81.26688300441795


In [20]:
import numpy as np

def compute_adjusted_life_expectancy(gender, death_params_df):
    """
    Life expectancy with adjusted mortality from age 40+:
    q*(age) = q(age) * exp(a * beta)
    with beta = 0.21 / 40
    """
    beta = 0
    max_age = 120

    # 1. q(a) for 0..85
    q = np.array([
        get_agent_death_prob(2021, gender, int(a), df=death_params_df)
        for a in range(0, 86)
    ])

    # 2. apply adjustment from age 40+
    q_adj = q.copy()
    for a in range(40, 86):     # apply for ages 40..85
        q_adj[a] = q[a] * (1- np.exp(a * 0.21/40)  * 0.15 -np.exp(a * 0.1/40) * 0.077 )/ 0.755

    # 3. extend 85+ using q_adj(85)
    q_extended = np.zeros(max_age + 1)
    q_extended[:86] = q_adj
    q_extended[86:] = q_adj[85]   # 86..120 same as 85

    # 4. survival curve
    s = np.zeros(max_age + 1)
    s[0] = 1.0
    for a in range(1, max_age + 1):
        s[a] = s[a - 1] * (1 - q_extended[a])

    # 5. life expectancy = sum of survival probabilities
    LE = np.sum(s)
    return LE


# ------------------------------------------------------------
# Chinese male & female LE with adjusted mortality
# ------------------------------------------------------------

LE_chinese_male_new   = compute_adjusted_life_expectancy("male",   death_params_df)
LE_chinese_female_new = compute_adjusted_life_expectancy("female", death_params_df)

print("Adjusted Chinese Male LE:   ", LE_chinese_male_new)
print("Adjusted Chinese Female LE: ", LE_chinese_female_new)


# ------------------------------------------------------------
# Chinese population in 2021 (0 and 1 indices)
# ------------------------------------------------------------

chi_male_pop   = result_matrix[0, :, 31].sum()
chi_female_pop = result_matrix[1, :, 31].sum()
N_tot_chi = chi_male_pop + chi_female_pop

# population-weighted overall Chinese LE
LE_chinese_overall_new = (
    chi_male_pop * LE_chinese_male_new +
    chi_female_pop * LE_chinese_female_new
) / N_tot_chi

print("Adjusted Overall Chinese Life Expectancy:", LE_chinese_overall_new)

Adjusted Chinese Male LE:    81.79921713477171
Adjusted Chinese Female LE:  86.96202875630458
Adjusted Overall Chinese Life Expectancy: 84.44901574300211
